# <font color="lightgreen">ETL Load - Carga de Datos a DuckDB</font>  🦆

##### Cargar las tablas del esquema estrella en una base de datos DuckDB para su posterior análisis y consulta.
---

### <font color="lightgreen">Librerías</font>

In [31]:
import duckdb
import pandas as pd
from pathlib import Path
import sys
from datetime import datetime


print("Librerías importadas correctamente")
print('Versión de duckdb:', duckdb.__version__)
print('Versión de pandas:', pd.__version__)


Librerías importadas correctamente
Versión de duckdb: 1.5.1
Versión de pandas: 3.0.1


### <font color="lightgreen">Configuración</font>

In [32]:
DATA_CURATED = Path("../data/curated")
DATABASE_DIR = Path("../database")
SCHEMA_FILE = DATABASE_DIR / "schema.sql"
DB_FILE = DATABASE_DIR / "technova.duckdb"
print(f"Configuración establecida:\n- DATA_CURATED: {DATA_CURATED}\n- DATABASE_DIR: {DATABASE_DIR}\n- SCHEMA_FILE: {SCHEMA_FILE}\n- DB_FILE: {DB_FILE}")


Configuración establecida:
- DATA_CURATED: ..\data\curated
- DATABASE_DIR: ..\database
- SCHEMA_FILE: ..\database\schema.sql
- DB_FILE: ..\database\technova.duckdb


### <font color="lightgreen">Funciones</font	>

In [33]:
def log(msg, level="INFO"):
    """Función simple de logging"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] [{level}] {msg}")


def ejecutar_schema(conn):
    """Ejecuta el archivo schema.sql"""
    if not SCHEMA_FILE.exists():
        log(f"Archivo schema no encontrado: {SCHEMA_FILE}", "ERROR")
        sys.exit(1)
    
    with open(SCHEMA_FILE, 'r', encoding='utf-8') as f:
        schema_sql = f.read()
    
    try:
        conn.execute(schema_sql)
        log("Schema ejecutado correctamente")
    except Exception as e:
        log(f"Error ejecutando schema: {e}", "ERROR")
        sys.exit(1)


def cargar_tabla(conn, tabla, csv_path, columnas):
    """Carga una tabla con full-refresh (DELETE + INSERT)"""
    if not csv_path.exists():
        log(f"  ⚠️  Archivo no encontrado: {csv_path}", "WARNING")
        return 0
    
    df = pd.read_csv(csv_path, encoding='utf-8')
    
    if df.empty:
        log(f"  ⚠️  Archivo vacío: {csv_path}", "WARNING")
        return 0
    
    # DELETE
    conn.execute(f"DELETE FROM {tabla}")
    
    # INSERT
    conn.register('temp_df', df)
    columnas_str = ', '.join(columnas)
    conn.execute(f"""
        INSERT INTO {tabla} ({columnas_str})
        SELECT {columnas_str} FROM temp_df
    """)
    
    resultado = conn.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()
    count = resultado[0]

    # count = conn.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    
    log(f"  ✅ {tabla}: {count:,} filas cargadas")
    return count


def verificar_fk(conn):
    """Verifica que no haya registros huérfanos en fact_monitoreo"""
    log("\n🔍 Verificando integridad de claves foráneas...")
    
    verificaciones = [
        ('fact_monitoreo', 'id_tiempo', 'dim_tiempo', 'id_tiempo'),
        ('fact_monitoreo', 'id_metrica', 'dim_metrica', 'id_metrica'),
        ('fact_monitoreo', 'id_area', 'dim_area', 'id_area'),
        ('dim_objetivo', 'id_metrica', 'dim_metrica', 'id_metrica')
    ]
    
    todo_ok = True
    
    for fact_tab, fk_col, dim_tab, pk_col in verificaciones:
        query = f"""
            SELECT COUNT(*) FROM {fact_tab} f
            LEFT JOIN {dim_tab} d ON f.{fk_col} = d.{pk_col}
            WHERE d.{pk_col} IS NULL
        """
        huerfanos = conn.execute(query).fetchone()[0]
        if huerfanos > 0:
            log(f"  ❌ {fact_tab}.{fk_col} -> {dim_tab}: {huerfanos} huérfanos", "ERROR")
            todo_ok = False
        else:
            log(f"  ✅ {fact_tab}.{fk_col} -> {dim_tab}: OK")
    
    return todo_ok


def guardar_metadata(conn, resultados):
    """Guarda el registro de la carga"""
    # Crear tabla con SERIAL (autoincremental)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS metadata_carga (
            id INTEGER PRIMARY KEY,
            fecha TIMESTAMP,
            tabla VARCHAR,
            filas INTEGER
        )
    """)
    
    # Insertar sin especificar ID (DuckDB no soporta autoincrement directamente)
    # Alternativa: usar SEQUENCE
    conn.execute("CREATE SEQUENCE IF NOT EXISTS seq_metadata START 1")
    
    for tabla, filas in resultados.items():
        conn.execute("""
            INSERT INTO metadata_carga (id, fecha, tabla, filas)
            VALUES (nextval('seq_metadata'), ?, ?, ?)
        """, [datetime.now(), tabla, filas])
    
    log("  📝 Metadatos guardados en metadata_carga")


### <font color="lightgreen">Main</font>

In [34]:
def main():
    log("=" * 60)
    log("🚀 INICIANDO CARGA A DUCKDB")
    log("=" * 60)
    
    # 1. Crear directorio si no existe
    DATABASE_DIR.mkdir(parents=True, exist_ok=True)
    
    # 2. Conectar a DuckDB
    conn = duckdb.connect(str(DB_FILE))
    log(f"✅ Conectado a: {DB_FILE}")
    
    # 3. Ejecutar schema.sql
    ejecutar_schema(conn)
    
    # 4. Definir tablas

    tablas = {
    'dim_tiempo': {
        'csv': DATA_CURATED / 'dim_tiempo.csv',
        'columnas': ['id_tiempo', 'fecha', 'anio', 'mes', 'dia', 
                    'trimestre', 'semana', 'nombre_mes', 'dia_semana', 'es_finde']
    },
    'dim_area': {
        'csv': DATA_CURATED / 'dim_area.csv',
        'columnas': ['id_area', 'nombre_area']
    },
    'dim_metrica': {
        'csv': DATA_CURATED / 'dim_metrica.csv', 
        'columnas': ['id_metrica', 'nombre', 'categoria', 'subcategoria', 'unidad', 
        'formula', 'prioridad', 'frecuencia_recomendada', 
        'tipo_agregacion', 'tendencia_deseada', 'descripcion']
    },
    'dim_objetivo': {   # NUEVA TABLA
        'csv': DATA_CURATED / 'dim_objetivo.csv',
        'columnas': ['id_objetivo', 'id_metrica', 'anio', 'valor_objetivo', 
                    'umbral_verde', 'umbral_amarillo', 'peso_relativo']
    },
    'fact_monitoreo': {
        'csv': DATA_CURATED / 'fact_monitoreo.csv',
        'columnas': ['id_monitoreo', 'id_tiempo', 'id_metrica', 'id_area', 'valor', 'fuente']
    },
    'dim_empleado': {
        'csv': DATA_CURATED / 'dim_empleado.csv',
        'columnas': ['id_empleado', 'nombre', 'area', 'genero', 'fecha_ingreso']
    },
    'dim_proveedor': {
        'csv': DATA_CURATED / 'dim_proveedor.csv',
        'columnas': ['id_proveedor', 'nombre_proveedor']
    }
}

    
    # 5. Cargar en orden específico
    log("\n📤 EJECUTANDO FULL-REFRESH...")
    resultados = {}
    orden_carga = ['dim_tiempo', 'dim_area', 'dim_metrica', 'dim_objetivo', 
            'dim_empleado', 'dim_proveedor', 'fact_monitoreo']

    for tabla in orden_carga:
        if tabla in tablas:
            log(f"\n📁 {tabla}")
            config = tablas[tabla]
            filas = cargar_tabla(conn, tabla, config['csv'], config['columnas'])
            resultados[tabla] = filas
        else:
            log(f"⚠️  Tabla {tabla} no encontrada", "WARNING")
    
    # 6. Verificar integridad FK
    fk_ok = verificar_fk(conn)

    # 7. Guardar metadata
    guardar_metadata(conn, resultados)
    
    # 8. Resumen final
    log("\n" + "=" * 60)
    log("✅ CARGA COMPLETADA")
    log("=" * 60)
    log("\n📊 RESUMEN DE FILAS CARGADAS:")
    
    for tabla, filas in resultados.items():
        log(f"   • {tabla}: {filas:,} filas")
    
    if not fk_ok:
        log("\n⚠️  ADVERTENCIA: Problemas de integridad referencial detectados", "WARNING")
    
    log(f"\n💾 Base de datos: {DB_FILE}")
    
    conn.close()


### <font color="lightgreen">Ejecución</font>

In [35]:
if __name__ == "__main__":
    main()

[2026-04-05 23:33:03] [INFO] ============================================================
[2026-04-05 23:33:03] [INFO] 🚀 INICIANDO CARGA A DUCKDB
[2026-04-05 23:33:03] [INFO] ============================================================
[2026-04-05 23:33:03] [INFO] ✅ Conectado a: ..\database\technova.duckdb
[2026-04-05 23:33:03] [INFO] Schema ejecutado correctamente
[2026-04-05 23:33:03] [INFO] 
📤 EJECUTANDO FULL-REFRESH...
[2026-04-05 23:33:03] [INFO] 
📁 dim_tiempo


ConstraintException: Constraint Error: Violates foreign key constraint because key "id_tiempo: 20180101" is still referenced by a foreign key in a different table. If this is an unexpected constraint violation, please refer to our foreign key limitations in the documentation